In [ ]:
from pyspark.sql.functions import (
    col, trim, when, max as spark_max, explode, lit, from_json
)

from delta.tables import DeltaTable

tabela_nome = "silver_categorias"

if spark.catalog.tableExists("controle_pipeline"):
        df_controle = spark.table("controle_pipeline").filter(col("tabela") == tabela_nome)

        if df_controle.count() > 0: 
            ultimo_processamento = df_controle.select("ultimo_ts").collect()[0][0]
        else:
            ultimo_processamento = None
else: 
    ultimo_processamento = None

In [ ]:
nome_tabela = "categorias"
caminho_bronze = f"Files/bronze/{nome_tabela}"
df_bronze = spark.read.format("delta").load(caminho_bronze)

if ultimo_processamento:
    df_bronze = df_bronze.filter(
        col("ingestion_ts") > ultimo_processamento
    )

if df_bronze.rdd.isEmpty():
    print("Sem dados novos para processar.")
    dbutils.notebook.exit("No data")

In [ ]:
df_silver = df_bronze

In [ ]:
for c, t in df_silver.dtypes:
    if t == "string":
        df_silver = df_silver.withColumn(
            c,
            when(trim(col(c)) == "", None)
            .otherwise(trim(col(c)))
        )

In [ ]:
df_silver = df_silver.dropDuplicates([
    "codigo",
    "empresa",
])

In [ ]:
if spark.catalog.tableExists("silver_categorias"):

    delta_table = DeltaTable.forName(spark, "silver_categorias")

    (delta_table.alias("t")
            .merge(
                df_silver.alias("s"),
                """
                t.codigo = s.codigo
                AND t.empresa = s.empresa
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()  
    )

else: 
    df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_categorias")

In [ ]:
novo_max_ts = df_bronze.agg(
    spark_max("ingestion_ts").alias("max_ts")).collect()[0]["max_ts"]

df_update = spark.createDataFrame(
    [(tabela_nome, novo_max_ts)],
    ["tabela", "ultimo_ts"]
)

if spark.catalog.tableExists("controle_pipeline"):

    delta_ctrl = DeltaTable.forName(spark, "controle_pipeline")

    (delta_ctrl.alias("t")
        .merge(
            df_update.alias("s"),
            "t.tabela = s.tabela"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else: 
    df_update.write.format("delta").mode("overwrite").saveAsTable("controle_pipeline")